In [1]:
import os
import pandas as pd
import re
from Bio import SeqIO
import gzip

In [2]:
pd.set_option('display.max_rows', None)  # Все строки
pd.set_option('display.max_columns', None)  # Все столбцы
pd.set_option('display.max_colwidth', None)

In [24]:
MetaVF_common_res = "/mnt/SSD7TB/RUNS/kozyreva_ai/AKKM_ONT_MGI/annotation/isolate2_VirF_nucl.csv"

In [25]:
w_d = "/mnt/SSD7TB/RUNS/kozyreva_ai/AKKM_ONT_MGI/annotation"

In [26]:
headers=['seqid','VFG_id','identity','al_len','mismatch','gaps','start','stop','x','target_len','eval','score']

In [27]:
MetaVF_df = pd.read_csv(MetaVF_common_res, sep="\t", header=None, names=headers)

In [28]:
MetaVF_df

,seqid,VFG_id,identity,al_len,mismatch,gaps,start,stop,x,target_len,eval,score
0,iso2_chr1_manual_curated,VFG043550(gb|NP_214954),71.223,417,106,11,1682293,1682702,736,1145,8.790000e-15,93.5


In [29]:
MetaVF_df['seqid'] = MetaVF_df['seqid'].apply(lambda x: x.split('lcl|')[1] if 'lcl|' in x else x)

In [30]:
MetaVF_df['VFG_id'] = MetaVF_df['VFG_id'].apply(lambda x: x.split('(')[0] if '(' in x else x)

In [31]:
MetaVF_df.head(3)

,seqid,VFG_id,identity,al_len,mismatch,gaps,start,stop,x,target_len,eval,score
0,iso2_chr1_manual_curated,VFG043550,71.223,417,106,11,1682293,1682702,736,1145,8.790000e-15,93.5


In [32]:
VF_meta1 = '/mnt/SSD4TB/PROJECTS/kozyreva_works/280325_Serr_reseq_MGI/hybrid/UNICYCLER_OUT/in_work/MetaVF/VFs.csv'
VF_meta1_df = pd.read_csv(VF_meta1, sep="\t", header=0)
#VF_meta1_df.head(10)

In [33]:
VF_meta2 = '/mnt/SSD4TB/PROJECTS/kozyreva_works/vir_factors_db/VFDB_setB_nt.fas'

In [34]:
def extract_fasta_headers(fasta_path):
    """
    Извлекает все заголовки из мульти-FASTA файла
    Поддерживает как обычные файлы, так и сжатые (.gz)
    Аргументы:
    fasta_path (str): путь к FASTA файлу (может быть .gz)
    Возвращает:
    list: список всех заголовков
    """
    headers = []
    
    open_func = gzip.open if fasta_path.endswith('.gz') else open
    open_mode = 'rt' if fasta_path.endswith('.gz') else 'r'
    
    with open_func(fasta_path, open_mode) as handle:
        for record in SeqIO.parse(handle, "fasta"):
            headers.append(record.description)
    
    return headers

In [35]:
headers_list = extract_fasta_headers(VF_meta2)

In [36]:
print(f"Найдено {len(headers_list)} заголовков")
print("5 заголовков:")
for header in headers_list[-5:]:
    print(header)

Найдено 32683 заголовков
5 заголовков:
VFG049454(gb|WP_012413437) (yapK) autotransporter protein YapK [YapK (VF0508) - Adherence (VFC0001)] [Yersinia pseudotuberculosis PB1/+]
VFG049455(gb|WP_012304691) (yapK) autotransporter protein YapK [YapK (VF0508) - Adherence (VFC0001)] [Yersinia pseudotuberculosis YPIII]
VFG049460(gb|WP_011193017) (yapV) autotransporter protein YapV [YapV (VF0507) - Adherence (VFC0001)] [Yersinia pseudotuberculosis IP 32953]
VFG049461(gb|WP_041175737) (yapV) autotransporter protein YapV [YapV (VF0507) - Adherence (VFC0001)] [Yersinia pseudotuberculosis IP 31758]
VFG049462(gb|WP_012413963) (yapV) autotransporter protein YapV [YapV (VF0507) - Adherence (VFC0001)] [Yersinia pseudotuberculosis PB1/+]


In [37]:
def parse_vfg_string(vfg_string):
    """Парсит строку VFG и возвращает словарь с извлеченными значениями"""
    result = {
        'VFG_id': None,
        'VF_id': None,
        'gene': None,
        'type': None,
        'Description': None,
        'source': None,
        'ref_prot': None,
        'VFC_id': None,
        'bulk': vfg_string
    }
    
    # 1. Извлекаем VFG_id (VFG037170)
    vfg_match = re.search(r'(VFG\d+)', vfg_string)
    if vfg_match:
        result['VFG_id'] = vfg_match.group(1)
    
    # 2. Извлекаем ref_prot (WP_001081754)
    ref_prot_match = re.search(r'\(gb\|([^)]+)\)', vfg_string)
    if ref_prot_match:
        result['ref_prot'] = ref_prot_match.group(1)
    
    # 3. Извлекаем gene (plc1)
   # gene_match = re.search(r'\(([^)]+)\)\s[^([]+\[', vfg_string)
   # if gene_match:
   #     result['gene'] = gene_match.group(1)
    gene_match = re.search(r'\(gb\|[^)]+\)\s\(([^)]+)\)', vfg_string)
    if gene_match:
        result['gene'] = gene_match.group(1)
    
    # 4. Извлекаем Description (phospholipase C)
    desc_match = re.search(r'\)\s([^[]+)', vfg_string.split(')', 2)[-1])
    if desc_match:
        result['Description'] = desc_match.group(1).strip()
    
    # 5. Извлекаем VF_id (VF0470)
    vf_match = re.search(r'VF(\d+)', vfg_string)
    if vf_match:
        result['VF_id'] = f"VF{vf_match.group(1)}"
    
    # 6. Извлекаем VFC_id (VFC0235)
    vfc_match = re.search(r'VFC(\d+)', vfg_string)
    if vfc_match:
        result['VFC_id'] = f"VFC{vfc_match.group(1)}"
    
    # 7. Извлекаем source (Acinetobacter baumannii 1656-2)
    source_match = re.search(r'\[([^]]+)\]$', vfg_string)
    if source_match:
        result['source'] = source_match.group(1)
    
    # 8. Извлекаем type (Exotoxin)
    type_match = re.search(r'-\s([^(]+)\s\(', vfg_string)
    if type_match:
        result['type'] = type_match.group(1).strip()
    
    return result

In [38]:
Meta_VF_metadata2 = pd.DataFrame(columns=[
    'VFG_id', 'ref_prot', 'gene', 'Description', 
    'VF_id', 'VFC_id', 'source', 'type', 'bulk'
])

In [39]:
parsed_data = []
parsed_data = [parse_vfg_string(s) for s in headers_list]
Meta_VF_metadata2 = pd.DataFrame(parsed_data)

In [40]:
Meta_VF_metadata2[Meta_VF_metadata2['gene']=='NMV_RS10945']

,VFG_id,VF_id,gene,type,Description,source,ref_prot,VFC_id,bulk
22309,VFG042732,VF0075,NMV_RS10945,Adherence,- Adherence (VFC0001)],Neisseria meningitidis 8013,WP_002218144,VFC0001,VFG042732(gb|WP_002218144) (NMV_RS10945) type IV pilin protein [Type IV pili (VF0075) - Adherence (VFC0001)] [Neisseria meningitidis 8013]


In [41]:
MetaVF_df.head(3)

,seqid,VFG_id,identity,al_len,mismatch,gaps,start,stop,x,target_len,eval,score
0,iso2_chr1_manual_curated,VFG043550,71.223,417,106,11,1682293,1682702,736,1145,8.790000e-15,93.5


In [42]:
merge_full = pd.merge(
    MetaVF_df,
    Meta_VF_metadata2,
    on=['VFG_id'],
    how='left'
)
merge_full.head(3)

,seqid,VFG_id,identity,al_len,mismatch,gaps,start,stop,x,target_len,eval,score,VF_id,gene,type,Description,source,ref_prot,VFC_id,bulk
0,iso2_chr1_manual_curated,VFG043550,71.223,417,106,11,1682293,1682702,736,1145,8.790000e-15,93.5,VF0866,groEL2,Adherence,- Adherence (VFC0001)],Mycobacterium tuberculosis H37Rv,NP_214954,VFC0001,VFG043550(gb|NP_214954) (groEL2) molecular chaperone GroEL [GroEL (VF0866) - Adherence (VFC0001)] [Mycobacterium tuberculosis H37Rv]


In [29]:
display(merge_full[(merge_full['seqid']=='C2_plasmid1')|(merge_full['seqid']=='C1_chromosome')])
#merge_full.to_csv(f"{w_d}/VF_SER_NUCL_wMeta.tsv", index=False, sep="\t")

,seqid,VFG_id,identity,al_len,mismatch,gaps,start,stop,x,target_len,eval,score,VF_id,gene,type,Description,source,ref_prot,VFC_id,bulk
109,C1_chromosome,VFG049139,81.195,3164,561,32,1210235,1213381,3147,1,0.000000e+00,2516.0,VF0568,acrB,Antimicrobial activity/Competitive advantage,- Antimicrobial activity/Competitive advantage (VFC0325)],Klebsiella pneumoniae 342,WP_008805437,VFC0325,VFG049139(gb|WP_008805437) (acrB) acriflavine resistance protein B [AcrAB (VF0568) - Antimicrobial activity/Competitive advantage (VFC0325)] [Klebsiella pneumoniae 342]
110,C1_chromosome,VFG049146,81.043,3165,564,34,1210235,1213381,3147,1,0.000000e+00,2488.0,VF0568,acrB,Antimicrobial activity/Competitive advantage,- Antimicrobial activity/Competitive advantage (VFC0325)],Klebsiella variicola At-22,WP_012968813,VFC0325,VFG049146(gb|WP_012968813) (acrB) acriflavine resistance protein B [AcrAB (VF0568) - Antimicrobial activity/Competitive advantage (VFC0325)] [Klebsiella variicola At-22]
111,C1_chromosome,VFG049146,82.646,582,95,6,3675557,3676135,979,1557,3.900000e-140,510.0,VF0568,acrB,Antimicrobial activity/Competitive advantage,- Antimicrobial activity/Competitive advantage (VFC0325)],Klebsiella variicola At-22,WP_012968813,VFC0325,VFG049146(gb|WP_012968813) (acrB) acriflavine resistance protein B [AcrAB (VF0568) - Antimicrobial activity/Competitive advantage (VFC0325)] [Klebsiella variicola At-22]
112,C1_chromosome,VFG049143,81.019,3161,572,26,1210235,1213381,3147,1,0.000000e+00,2488.0,VF0568,acrB,Antimicrobial activity/Competitive advantage,- Antimicrobial activity/Competitive advantage (VFC0325)],Klebsiella pneumoniae subsp. pneumoniae MGH 78578,WP_002892069,VFC0325,VFG049143(gb|WP_002892069) (acrB) acriflavine resistance protein B [AcrAB (VF0568) - Antimicrobial activity/Competitive advantage (VFC0325)] [Klebsiella pneumoniae subsp. pneumoniae MGH 78578]
113,C1_chromosome,VFG049143,82.268,485,82,4,2441755,2442237,1015,1497,8.750000e-112,416.0,VF0568,acrB,Antimicrobial activity/Competitive advantage,- Antimicrobial activity/Competitive advantage (VFC0325)],Klebsiella pneumoniae subsp. pneumoniae MGH 78578,WP_002892069,VFC0325,VFG049143(gb|WP_002892069) (acrB) acriflavine resistance protein B [AcrAB (VF0568) - Antimicrobial activity/Competitive advantage (VFC0325)] [Klebsiella pneumoniae subsp. pneumoniae MGH 78578]
114,C2_plasmid1,VFG049248,100.000,62,0,0,9406,9467,1,62,5.590000e-23,115.0,VF0784,KPK_RS28635,Effector delivery system,- Effector delivery system (VFC0086)],Klebsiella pneumoniae 342,WP_123809090,VFC0086,VFG049248(gb|WP_123809090) (KPK_RS28635) peptidoglycan DD-metalloendopeptidase family protein [T6SS-III (VF0784) - Effector delivery system (VFC0086)] [Klebsiella pneumoniae 342]
115,C2_plasmid1,VFG048839,100.000,32,0,0,8408,8439,1349,1380,2.660000e-06,60.2,VF0560,wcaJ,Immune modulation,- Immune modulation (VFC0258)],Klebsiella pneumoniae KCTC 2242,WP_041937831,VFC0258,VFG048839(gb|WP_041937831) (wcaJ) undecaprenyl-phosphate glucose phosphotransferase [Capsule (VF0560) - Immune modulation (VFC0258)] [Klebsiella pneumoniae KCTC 2242]


In [43]:
print(f"{w_d}/VF_isolate2_NUCL_wMeta.tsv")

/mnt/SSD7TB/RUNS/kozyreva_ai/AKKM_ONT_MGI/annotation/VF_isolate2_NUCL_wMeta.tsv


In [44]:
merge_full.to_csv(f"{w_d}/VF_isolate2_NUCL_wMeta.tsv", index=False, sep="\t")